In [ ]:
import fastf1
import pandas as pd

In [ ]:
schedule_df = pd.read_csv('./events/events.csv')
sessions = ['Session2', 'Session3', 'Session4', 'Session5']
year = 2026
races = ['United States Grand Prix', 'Mexico City Grand Prix'] 
failed_races = ''
for race in races:
    race_df = schedule_df.loc[(schedule_df['EventName'] == race) & (schedule_df['Year'] == year)].reset_index(drop=True)
    for session in sessions:
        session_type = race_df[session].iloc[0]
        if session_type in ['Race', 'Sprint', 'Qualifying', 'Sprint Qualifying', 'Sprint Shootout']:
            try:
                session_data = fastf1.get_session(year, race, session_type)
                session_data.load()

                session_type = session_data.session_info['Type']
                name = f'{year}_{race}_{session_type}'
                results = session_data.results
                laps = session_data.laps
                weather = session_data.weather_data

                # exporting dfs to parquets

                results.to_parquet(f'./data/{name}_results.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')
                laps.to_parquet(f'./data/{name}_laps.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')
                weather.to_parquet(f'./data/{name}_weather.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')

            except:
                failed_races = failed_races + f'<{race} - {session} Failed> '


failed_races
